In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as psf
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

spark: SparkSession = (
    SparkSession.builder.config(
        "spark.driver.extraJavaOptions", "-Djava.security.manager=allow"
    )
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow")
    .getOrCreate()
)

In [ ]:
df_batch_measurements = spark.read.parquet("../data/batch_measurements.parquet")
df_measurements = spark.read.parquet("../data/measurements.parquet")


def clean(df: DataFrame) -> DataFrame:
    return df.withColumn("name", psf.initcap(psf.col("name")))

When was the first and last measurement of each bear taken?


In [ ]:
(
    df_measurements.groupBy("name")
    .agg(
        psf.min("timestamp").alias("first_measurement"),
        psf.max("timestamp").alias("last_measurement"),
    )
    .show(10, False)
)

For each lifestage group of polar bears and for each year, which polar bear was the most active (most amount of steps per day)?


In [ ]:
(
    df_batch_measurements.transform(clean)
    .withColumn("year", psf.year(psf.col("timestamp")))
    .withColumn(
        "max_daily_steps",
        psf.max("daily_steps").over(Window.partitionBy("year", "life_stage")),
    )
    .filter(psf.col("daily_steps") == psf.col("max_daily_steps"))
    .select("life_stage", "year", "name", "daily_steps")
    .sort("year", "life_stage")
    .show()
)

For every year, figure out which polar bear was the heaviest.


In [ ]:
(
    df_batch_measurements.transform(clean)
    .withColumn("year", psf.year(psf.col("timestamp")))
    .withColumn("max_weight", psf.max("weight").over(Window.partitionBy("year")))
    .filter(psf.col("weight") == psf.col("max_weight"))
    .select("year", "name", "weight")
    .show()
)

Find out which bears were more anxious (higher blood pressure) than average the day after New Year's Eve (probably because of fireworks)?


In [ ]:
(
    df_measurements
    # Let's use the average of the blood pressure per polar bear as the baseline
    .withColumn(
        "avg_blood_pressure", psf.avg("blood_pressure").over(Window.partitionBy("name"))
    )
    .filter(
        (psf.month(psf.col("timestamp")) == 1) & (psf.day(psf.col("timestamp")) == 2)
    )
    .filter(psf.col("blood_pressure") > psf.col("avg_blood_pressure"))
    .select("name", psf.year(psf.col("timestamp")).alias("year"))
    .distinct()
    .sort("year", "name")
    .show()
)